In [1]:
# To create reproducible file paths
import os
from pathlib import Path
import pathlib

In [2]:
# To use APIs
import requests

# To download the GBIF data
from getpass import getpass
import pygbif.occurrences as occ
import pygbif.species as species

In [3]:
# For unzipping folders
import time
import zipfile

In [4]:
# To work with different types of data
import geopandas as gpd # To make GeoDataFrames/work with vector data
from glob import glob # To combine data arrays
import numpy as np # To work with arrays
import pandas as pd # To work with dataframes
import rioxarray as rxr # To work with raster data

In [5]:
import rioxarray.merge as rxrm # To merge raster data
from shapely.geometry import MultiPolygon, Polygon # To handle invalid geometries
import xrspatial # To handle spatial data

In [6]:
# For visualization and interactive plotting
import geoviews as gv
import holoviews as hv
import hvplot.pandas
import hvplot.xarray

In [7]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from shapely.geometry import MultiPolygon, Polygon # To handle invalid geometries

In [ ]:
# Import the below libraries:
# To create reproducible file paths
import os
from pathlib import Path
import pathlib

# To use APIs
import earthaccess # For logging into NASA Earth Access
import requests

# To download the GBIF data
from getpass import getpass
import pygbif.occurrences as occ
import pygbif.species as species

# For unzipping folders
import time
import zipfile

# To work with different types of data
import geopandas as gpd # To make GeoDataFrames/work with vector data
from glob import glob # To combine data arrays
from math import floor, ceil # For dealing with integers
import numpy as np # To work with arrays
import pandas as pd # To work with dataframes
import rioxarray as rxr # To work with raster data
import rioxarray.merge as rxrm # To merge raster data
from shapely.geometry import MultiPolygon, Polygon # To handle invalid geometries
import xarray as xr # To use xarray datasets
import xrspatial # To handle spatial data

# For visualization and interactive plotting
import geoviews as gv
import holoviews as hv
import hvplot.pandas
import hvplot.xarray
import matplotlib.gridspec as gridspec 
import matplotlib.pyplot as plt

In [29]:
# Create a designated folder for the repository data
moss_data_dir = os.path.join(
    pathlib.Path.home(),
    # In the earth-analytics data folder
    'earth-analytics',
    'data',
    # Specify the destination as inside the "spring-03-habitat-suitability-climate-change-_____" repository
    'spring-03-habitat-suitability-climate-change-Livian-Von-Dran',
    'spreading_earthmoss_habitat_suitability'
)

# Create the directory
os.makedirs(moss_data_dir, exist_ok=True)

In [30]:
# Create a directory for the GBIF data
gbif_moss_dir = os.path.join(moss_data_dir, 'gbif_moss_dir')

In [31]:
# Permanently log into GBIF
# Do not reset credentials to avoid repeated login requests
reset_credentials = False

# Request and store username
if (not ('GBIF_USER'  in os.environ)) or reset_credentials:
    os.environ['GBIF_USER'] = input('GBIF username:')

# Request and store password
if (not ('GBIF_PWD'  in os.environ)) or reset_credentials:
    os.environ['GBIF_PWD'] = getpass('GBIF password:')
    
# Request and store email address
if (not ('GBIF_EMAIL'  in os.environ)) or reset_credentials:
    os.environ['GBIF_EMAIL'] = input('GBIF email:')

In [34]:
# Check that the login attempt was successful
'GBIF_PWD' in os.environ

True

In [35]:
# Set the species name
species_name = "Physcomitrella patens"

# Pull the species info from GBIF
species_info = species.name_lookup(species_name, rank = 'Species')

# Grab the first result and print it
first_result = species_info['results'][0]
first_result

{'key': 100540353,
 'datasetKey': '39653f3e-8d6b-4a94-a202-859359c164c5',
 'nubKey': 2674024,
 'parentKey': 100540352,
 'parent': 'Physcomitrella',
 'kingdom': 'Plantae',
 'phylum': 'Bryophyta',
 'order': 'Funariales',
 'family': 'Funariaceae',
 'genus': 'Physcomitrella',
 'species': 'Physcomitrella patens',
 'kingdomKey': 100539876,
 'phylumKey': 100539878,
 'classKey': 100539913,
 'orderKey': 100540335,
 'familyKey': 100540339,
 'genusKey': 100540352,
 'speciesKey': 100540353,
 'scientificName': 'Physcomitrella patens (Hedw.) Schimp.',
 'canonicalName': 'Physcomitrella patens',
 'authorship': '(Hedw.) Schimp.',
 'nameType': 'SCIENTIFIC',
 'taxonomicStatus': 'ACCEPTED',
 'rank': 'SPECIES',
 'origin': 'SOURCE',
 'numDescendants': 0,
 'numOccurrences': 0,
 'taxonID': '23117',
 'habitats': [],
 'nomenclaturalStatus': [],
 'threatStatuses': [],
 'descriptions': [],
 'vernacularNames': [{'vernacularName': 'slibmos', 'language': 'nld'}],
 'synonym': False,
 'higherClassificationMap': {'1005

In [36]:
# Get the species key
species_key = first_result['nubKey']

# Check what the species key is
first_result['species'], species_key

('Physcomitrella patens', 2674024)

In [14]:
# Assign the species key found in the previous inquiry
species_key = 2674024

In [ ]:
# Create the file path
gbif_moss_pattern = os.path.join(gbif_moss_dir, '*.csv')

# Create a function to download the moss occurrence data
if not glob(gbif_moss_pattern):
    # Submit the query to GBIF
    gbif_query = occ.download([
        f"speciesKey = {species_key}",
        "hasCoordinate = True",
    ])
    # Only download the data once
    if not 'GBIF_DOWNLOAD_KEY' in os.environ:
        os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0]
        download_key = os.environ['GBIF_DOWNLOAD_KEY']
        time.sleep(5)
    # Download the data
    download_info = occ.download_get(
        os.environ['GBIF_DOWNLOAD_KEY'],
        path = moss_data_dir
    )
    # Unzip the file
    with zipfile.ZipFile(download_info['path']) as download_zip:
        download_zip.extractall(path = gbif_moss_dir)

# Locate the CSV file path
gbif_path = glob(gbif_moss_pattern)[0]

In [21]:
# Look at the download information
occ.download_meta("0063249-260226173443078") # Input the download key to view the information

{'key': '0063249-260226173443078',
 'doi': '10.15468/dl.w2rwmv',
 'license': 'unspecified',
 'request': {'predicate': {'type': 'and',
   'predicates': [{'type': 'equals',
     'key': 'SPECIES_KEY',
     'value': '2674024',
     'matchCase': False},
    {'type': 'equals',
     'key': 'HAS_COORDINATE',
     'value': 'True',
     'matchCase': False}]},
  'sendNotification': True,
  'format': 'SIMPLE_CSV',
  'type': 'OCCURRENCE',
  'verbatimExtensions': [],
  'interpretedExtensions': []},
 'created': '2026-03-26T16:53:33.494+00:00',
 'modified': '2026-03-26T17:43:50.108+00:00',
 'eraseAfter': '2026-09-26T16:53:33.350+00:00',
 'status': 'SUCCEEDED',
 'downloadLink': 'https://api.gbif.org/v1/occurrence/download/request/0063249-260226173443078.zip',
 'size': 516,
 'totalRecords': 0,
 'numberDatasets': 0}

In [23]:
# Read the CSV
gbif_moss_df = pd.read_csv(
    gbif_path,
    delimiter = '\t'
)

# Check the dataframe
gbif_moss_df

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue


In [24]:
# Print the columns
gbif_moss_df.columns

Index(['gbifID', 'datasetKey', 'occurrenceID', 'kingdom', 'phylum', 'class',
       'order', 'family', 'genus', 'species', 'infraspecificEpithet',
       'taxonRank', 'scientificName', 'verbatimScientificName',
       'verbatimScientificNameAuthorship', 'countryCode', 'locality',
       'stateProvince', 'occurrenceStatus', 'individualCount',
       'publishingOrgKey', 'decimalLatitude', 'decimalLongitude',
       'coordinateUncertaintyInMeters', 'coordinatePrecision', 'elevation',
       'elevationAccuracy', 'depth', 'depthAccuracy', 'eventDate', 'day',
       'month', 'year', 'taxonKey', 'speciesKey', 'basisOfRecord',
       'institutionCode', 'collectionCode', 'catalogNumber', 'recordNumber',
       'identifiedBy', 'dateIdentified', 'license', 'rightsHolder',
       'recordedBy', 'typeStatus', 'establishmentMeans', 'lastInterpreted',
       'mediaType', 'issue'],
      dtype='str')

In [26]:
# Convert the dataframe into a geodataframe (GDF)
gbif_moss_gdf = (
    gpd.GeoDataFrame(
        gbif_moss_df,
        # Add geometry columns to convert to a GDF
        geometry = gpd.points_from_xy(
            gbif_moss_df.decimalLongitude,
            gbif_moss_df.decimalLatitude
        ),
        crs = 'EPSG:4326'
    )
)

# Display the GDF data
gbif_moss_gdf

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue,geometry


In [ ]:
# Create an interactive plot of the GDIF moss observation data
gbif_moss_gdf.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Spreading Earthmoss Observations on GDIF',
    # Avoid using a fill color, but select a line color of your choice
    fill_color = None,
    line_color = 'orange',
)